# ⚡ AegisX — Train on Kaggle (free GPU P100/T4)

Kaggle version of the training notebook. **No Google Drive** - everything
lives under `/kaggle/working/` and you **download the export ZIP at the end**
(Kaggle sessions can end at any time, so download before closing).

**Setup in Kaggle:** Settings (gear icon) -> Accelerator **GPU P100** and
Internet **ON**. Then File > Upload Notebook, or copy-paste cells.

**Your laptop never trains anything.** All compute happens on Kaggle's free GPU.

## 1. Setup

In [ ]:
!pip install -q torch

import os
import sys
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('Kaggle working dir:', os.getcwd())

## 2. Get the AegisX code

Clones the repo into `/kaggle/working/` (Kaggle has no Drive; results stay
in the working dir until you download them).

In [ ]:
WORK = '/kaggle/working/aegisx'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

if not os.path.isdir('.git'):
    !git clone https://github.com/FerzDevZ/AegisX.git .
else:
    !git -C . pull --ff-only
print('Repo ready:')
print(os.listdir(WORK))

## 2b. Optional: download MORE training data (recommended)

Pulls OWASP cheat sheets, ASVS, MITRE ATT&CK + ~10k recent CVEs into
`data/raw/`. Needs Internet ON (Settings). Each file downloads individually,
so a single failure does not stop the rest.

In [ ]:
DOWNLOAD_MORE_DATA = True
if DOWNLOAD_MORE_DATA:
    !python scripts/fetch_corpus.py --target-dir data/raw --cve-pages 5
else:
    print('Skipped corpus download.')

total = sum(os.path.getsize(os.path.join('data/raw', f)) for f in os.listdir('data/raw') if f.endswith('.txt'))
print(f'Corpus now: {total/1024:.0f} KB of text')

## 3. Configure training

In [ ]:
# --- training hyperparameters (same as Colab version) ---
DATA_DIR      = 'data/raw'
OUT_DIR       = '/kaggle/working/checkpoints/aegisx-mini'
VOCAB_SIZE    = 4096
BLOCK_SIZE    = 256
N_LAYER       = 8
N_HEAD        = 8
N_EMBD        = 512
BATCH_SIZE    = 16
GRAD_ACCUM    = 4
MAX_STEPS     = 6000
LR            = 3e-4
WARMUP_STEPS  = 300
EVAL_EVERY    = 300
EARLY_STOP    = 5
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 4. Train

AMP is auto-enabled on CUDA (~2x faster). Watch for `🛑 early stop` - it
should fire well before 6000 steps on this corpus.

In [ ]:
!python -m aegisx.train \
    --data {DATA_DIR} \
    --out {OUT_DIR} \
    --vocab-size {VOCAB_SIZE} \
    --block-size {BLOCK_SIZE} \
    --n-layer {N_LAYER} \
    --n-head {N_HEAD} \
    --n-embd {N_EMBD} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --max-steps {MAX_STEPS} \
    --lr {LR} \
    --warmup-steps {WARMUP_STEPS} \
    --eval-every {EVAL_EVERY} \
    --early-stop-patience {EARLY_STOP} \
    --device {DEVICE}

## 5. Quick sanity check

In [ ]:
import os
if os.path.exists(f'{OUT_DIR}/model.pt'):
    !python -m aegisx.chat --model {OUT_DIR}/model.pt --tokenizer {OUT_DIR}/tokenizer.json \
        --prompt "You are AegisX, a cybersecurity assistant. User: how do I enumerate subdomains?\n\nAegisX:" \
        --max-new-tokens 150 --temperature 0.8 --top-k 50
else:
    print('Skipped: model.pt not found - check the training cell output for errors.')

## 5b. Evaluate (before you upload)

Runs 20 fixed questions (10 EN + 10 ID) and prints keyword-coverage scores.
Use this to judge whether the model is worth uploading and to compare retrains.

In [ ]:
import os
if os.path.exists(f'{OUT_DIR}/model.pt'):
    !python -m aegisx.eval --model {OUT_DIR}/model.pt --tokenizer {OUT_DIR}/tokenizer.json \
        --device {DEVICE} --max-new-tokens 90
else:
    print('Skipped: model.pt not found - check the training cell output for errors.')

## 6. Package for manual Hugging Face upload

Copies model + tokenizer + config + model card + `knowledge/` into an export
folder and ZIPs it. **No auto-push** - you download and upload manually.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

if not os.path.exists(f'{OUT_DIR}/model.pt'):
    print('Skipped: model.pt not found - nothing to export. Train first.')
else:
    EXPORT_DIR = Path('/kaggle/working/export/aegisx-mini')
    if EXPORT_DIR.exists():
        shutil.rmtree(EXPORT_DIR)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copy(f'{OUT_DIR}/model.pt', EXPORT_DIR / 'model.pt')
    shutil.copy(f'{OUT_DIR}/tokenizer.json', EXPORT_DIR / 'tokenizer.json')
    shutil.copy(f'{OUT_DIR}/config.json', EXPORT_DIR / 'config.json')

    card_src = Path('hf/MODEL_CARD.md')
    if card_src.exists():
        shutil.copy(card_src, EXPORT_DIR / 'README.md')

    KNOW_DIR = EXPORT_DIR / 'knowledge'
    KNOW_DIR.mkdir(parents=True, exist_ok=True)
    n_know = 0
    for f in sorted(Path('data/raw').glob('*.txt')):
        shutil.copy(f, KNOW_DIR / f.name)
        n_know += 1
    print(f'  knowledge/ folder: {n_know} files (RAG grounding)')

    zip_path = Path(str(EXPORT_DIR) + '.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(EXPORT_DIR.iterdir()):
            if f.is_dir():
                for inner in sorted(f.rglob('*')):
                    if inner.is_file():
                        zf.write(inner, arcname=f'{f.name}/{inner.name}')
            else:
                zf.write(f, arcname=f.name)

    print('Export folder:')
    for f in sorted(EXPORT_DIR.iterdir()):
        if f.is_dir():
            print(f'  {f.name}/')
        else:
            print(f'  {f.name}  ({f.stat().st_size:,} bytes)')
    print(f'ZIP: {zip_path}')

## 7. ⚠️ DOWNLOAD the export NOW

Kaggle sessions end without warning. **Run the cell below and click the
download link immediately** - then upload the ZIP manually to Hugging Face.

In [ ]:
import os
zip_path = '/kaggle/working/export/aegisx-mini.zip'
if os.path.exists(zip_path):
    print('ZIP ready. Click the link below to download:')
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
else:
    print('ZIP not found - run cell 6 first.')